In [1]:
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt

In [ ]:
def show_cluster(lat, lon, meters_around, time):
    graph = ox.graph_from_point((lat, lon), dist=meters_around, network_type='drive')
    graph = ox.add_edge_speeds(graph)
    graph = ox.add_edge_travel_times(graph)

    simetric_graph = nx.Graph()

    for u, v, data in graph.edges(data=True):
        tiempo = data.get('travel_time', 10) # 10s por defecto si no hay dato
        
        # Si la arista ya existe en sentido contrario, promediamos el peso
        if simetric_graph.has_edge(u, v):
            peso_actual = simetric_graph[u][v]['peso_tda']
            # Aquí aplicas la misma ponderación que usaste en tu matriz original
            simetric_graph[u][v]['peso_tda'] = (peso_actual + tiempo) / 2
        else:
            # Si no existe, la creamos y le pasamos su geometría original para que se dibuje bien
            simetric_graph.add_edge(u, v, weight=tiempo, peso_tda=tiempo, geometry=data.get('geometry'))

    nodo_origen = ox.distance.nearest_nodes(graph, X=long, Y=lat)

    subgrafo_cluster = nx.ego_graph(
        simetric_graph, 
        nodo_origen, 
        radius=time, 
        distance='peso_tda'
    )

    print(f"Subgrafo extraído: {len(subgrafo_cluster.nodes)} nodos encontrados en el clúster.")

    #Map rendering

    ig, ax = ox.plot_graph(
        graph, 
        node_size=0, 
        edge_color="#333333", 
        edge_linewidth=0.5, 
        edge_alpha=0.2, 
        show=False, 
        close=False, 
        bgcolor="#111111"
    )

    subgrafo_render = nx.MultiDiGraph(subgrafo_cluster)
    fig, ax = ox.plot_graph(
        subgrafo_render, 
        ax=ax, 
        node_size=10, 
        node_color="cyan", 
        edge_color="cyan", 
        edge_linewidth=2, 
        edge_alpha=0.8, 
        show=True
    )

    


In [ ]:
lat = 19.28659873
long = -99.03266315

km_around = 3000

time = 300

show_cluster(lat, long, km_around, time)